In [ ]:
import geopandas as gpd
import pandas as pd
import duckdb

# setup bucket access
bucket = 's3://EarthCODE/'
endpoint_url = "https://s3.waw4-1.cloudferro.com"
region_name = "eu-west-2"
prefix = 'OSCAssets/storm-data/'

## earthcare_id -> file mapping
mapping = pd.read_parquet('earthcare_id_mapping.parquet').set_index('earthcare_id')
mapping

,files
earthcare_id,
01253H,[GLM_2024_8.parquet]
01042F,[GLM_2024_8.parquet]
01206H,[GLM_2024_8.parquet]
01214F,[GLM_2024_8.parquet]
01283H,[GLM_2024_8.parquet]
...,...
09064F,[LI_2026_1.parquet]
09290H,[LI_2026_1.parquet]
09398H,[LI_2026_1.parquet]


# 

In [4]:
# setup example files  and filters
earthcare_id = '01206H'
earthcare_id_file = mapping.loc[earthcare_id].values[0][0]
print(earthcare_id_file)

file = 'LI_2024_8.parquet'
example_bbox = (13.3, 34.8, 29.7, 48.3)

GLM_2024_8.parquet


# Direct Geopandas


## Read the full dataset

In [ ]:
gdf = gpd.read_parquet(
    f"{bucket}{prefix}{file}",
    storage_options={ "anon": True, 
                    "client_kwargs": {
                        "endpoint_url": endpoint_url,
                        "region_name": region_name
                    }
    }
)
gdf.to_parquet(f'{file}.parquet')

## Read only a spatial subset of the data

In [ ]:
gdf = gpd.read_parquet(
    f"{bucket}{prefix}{file}",
    storage_options={ "anon": True, 
                    "client_kwargs": {
                        "endpoint_url": endpoint_url,
                        "region_name": region_name
                    }
    },
bbox=example_bbox
)
gdf

,group_id,group_time,latitude,longitude,radiance,flash_id,parallax_corrected_lat,parallax_corrected_lon,ec_time_diff,parent_cluster_id,cluster_id,distance_from_nadir,earthcare_id,geometry
0,141812743,2024-08-31 12:08:01.794299904,37.389603,29.686501,54.5,20754348,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,POINT (29.6865 37.3896)
1,141812744,2024-08-31 12:08:01.795299968,37.392300,29.686501,84.0,20754348,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,POINT (29.6865 37.3923)
2,143179852,2024-08-31 13:33:33.015499904,37.419300,29.697300,7.0,21165661,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,POINT (29.6973 37.4193)
3,142675272,2024-08-31 13:04:25.143600000,37.635300,29.624401,14.5,21028628,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,POINT (29.6244 37.6353)
4,142619763,2024-08-31 13:00:57.755399936,37.632603,29.624401,4.0,21012347,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,POINT (29.6244 37.6326)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564972,76049464,2024-08-09 13:30:06.583799936,46.842300,28.611900,47.0,18038280,NaN,NaN,-9223372036854775808,58,NaN,NaN,01129D,POINT (28.6119 46.8423)
564973,76219367,2024-08-09 13:39:25.464599936,46.869301,28.601101,8.5,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,POINT (28.6011 46.8693)
564974,76219368,2024-08-09 13:39:25.475600000,46.863903,28.598400,20.5,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,POINT (28.5984 46.8639)
564975,76219373,2024-08-09 13:39:25.505600000,46.866600,28.595701,10.0,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,POINT (28.5957 46.8666)


### Filter by earthcare id

In [ ]:
gdf = gpd.read_parquet(
    f"{bucket}{prefix}{earthcare_id_file}",
    storage_options={ "anon": True, 
                    "client_kwargs": {
                        "endpoint_url": endpoint_url,
                        "region_name": region_name
                    }
    },
    filters=[('earthcare_id', "==", earthcare_id)],
)
gdf

# Duckdb

## Spatial query

In [ ]:
# Configure DuckDB for your S3 endpoint
duckdb.sql("INSTALL httpfs; LOAD httpfs; INSTALL spatial; LOAD spatial;")
duckdb.sql(f"SET s3_endpoint='{endpoint_url.replace('https://', '')}';")
duckdb.sql(f"SET s3_region='{region_name}';")

file_path = f"s3://{bucket.replace('s3://', '')}{prefix}{file}"

In [ ]:
query = f"""
SELECT 
    * EXCLUDE (geometry), 
    ST_AsWKB(geometry) AS geometry
    FROM read_parquet('{file_path}') 
    WHERE longitude >= {example_bbox[0]} 
      AND longitude <= {example_bbox[2]}
      AND latitude >= {example_bbox[1]} 
      AND latitude <= {example_bbox[3]} 
"""

# Fetch as an Arrow table, then convert to GeoPandas
arrow_stream = duckdb.sql(query).arrow()
arrow_table = arrow_stream.read_all()
df = arrow_table.to_pandas()
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))
gdf

,group_id,group_time,latitude,longitude,radiance,flash_id,parallax_corrected_lat,parallax_corrected_lon,ec_time_diff,parent_cluster_id,cluster_id,distance_from_nadir,earthcare_id,bbox,geometry
0,141812743,2024-08-31 12:08:01.794299904,37.389603,29.686501,54.5,20754348,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,"{'xmin': 29.686500549316406, 'ymin': 37.389602...",POINT (29.6865 37.3896)
1,141812744,2024-08-31 12:08:01.795299968,37.392300,29.686501,84.0,20754348,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,"{'xmin': 29.686500549316406, 'ymin': 37.392299...",POINT (29.6865 37.3923)
2,143179852,2024-08-31 13:33:33.015499904,37.419300,29.697300,7.0,21165661,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,"{'xmin': 29.69729995727539, 'ymin': 37.4193000...",POINT (29.6973 37.4193)
3,142675272,2024-08-31 13:04:25.143600000,37.635300,29.624401,14.5,21028628,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,"{'xmin': 29.624401092529297, 'ymin': 37.635299...",POINT (29.6244 37.6353)
4,142619763,2024-08-31 13:00:57.755399936,37.632603,29.624401,4.0,21012347,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01471D,"{'xmin': 29.624401092529297, 'ymin': 37.632602...",POINT (29.6244 37.6326)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564972,76049464,2024-08-09 13:30:06.583799936,46.842300,28.611900,47.0,18038280,NaN,NaN,-9223372036854775808,58,NaN,NaN,01129D,"{'xmin': 28.611900329589844, 'ymin': 46.842300...",POINT (28.6119 46.8423)
564973,76219367,2024-08-09 13:39:25.464599936,46.869301,28.601101,8.5,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,"{'xmin': 28.60110092163086, 'ymin': 46.8693008...",POINT (28.6011 46.8693)
564974,76219368,2024-08-09 13:39:25.475600000,46.863903,28.598400,20.5,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,"{'xmin': 28.598400115966797, 'ymin': 46.863903...",POINT (28.5984 46.8639)
564975,76219373,2024-08-09 13:39:25.505600000,46.866600,28.595701,10.0,18101636,NaN,NaN,-9223372036854775808,-1,NaN,NaN,01129D,"{'xmin': 28.595701217651367, 'ymin': 46.866600...",POINT (28.5957 46.8666)


### Column filtering

In [7]:
file_path = f"s3://{bucket.replace('s3://', '')}{prefix}{earthcare_id_file}"

query = f"""
    SELECT 
    * EXCLUDE (geometry), 
    ST_AsWKB(geometry) AS geometry
    FROM read_parquet('{file_path}') 
    WHERE earthcare_id = '{earthcare_id}'
"""

# Fetch as an Arrow table, then convert to GeoPandas
arrow_stream = duckdb.sql(query).arrow()
arrow_table = arrow_stream.read_all()
df = arrow_table.to_pandas()
gdf = gpd.GeoDataFrame(df, geometry=gpd.GeoSeries.from_wkb(df['geometry']))

gdf

,group_id,group_time,latitude,longitude,radiance,flash_id,ec_time_diff,parent_cluster_id,cluster_id,distance_from_nadir,earthcare_id,bbox,geometry
0,62909809,2024-08-14 13:05:40.140002304,-38.556141,-151.038437,2.584874e-15,30010,-9223372036854775808,0,NaN,NaN,01206H,"{'xmin': -151.03843688964844, 'ymin': -38.5561...",POINT (-151.03844 -38.55614)
1,62909811,2024-08-14 13:05:40.143816960,-38.559448,-151.024841,7.484286e-15,30010,-9223372036854775808,0,NaN,NaN,01206H,"{'xmin': -151.02484130859375, 'ymin': -38.5594...",POINT (-151.02484 -38.55945)
2,62909810,2024-08-14 13:05:40.141909632,-38.562645,-151.025955,1.178377e-14,30010,-9223372036854775808,0,NaN,NaN,01206H,"{'xmin': -151.0259552001953, 'ymin': -38.56264...",POINT (-151.02596 -38.56264)
3,62909784,2024-08-14 13:05:40.091936640,-38.591560,-151.022903,4.384658e-15,30010,-9223372036854775808,0,NaN,NaN,01206H,"{'xmin': -151.0229034423828, 'ymin': -38.59156...",POINT (-151.0229 -38.59156)
4,62909812,2024-08-14 13:05:40.145724288,-38.598660,-151.037582,6.457743e-14,30010,-9223372036854775808,0,NaN,NaN,01206H,"{'xmin': -151.03758239746094, 'ymin': -38.5986...",POINT (-151.03758 -38.59866)
...,...,...,...,...,...,...,...,...,...,...,...,...,...
10660,62980126,2024-08-14 13:25:32.338066048,-36.837769,-149.900665,1.185042e-15,35679,-9223372036854775808,7,NaN,NaN,01206H,"{'xmin': -149.90066528320312, 'ymin': -36.8377...",POINT (-149.90067 -36.83777)
10661,62977261,2024-08-14 13:24:37.201879552,-36.837677,-149.900681,6.851020e-16,35426,-9223372036854775808,7,NaN,NaN,01206H,"{'xmin': -149.9006805419922, 'ymin': -36.83767...",POINT (-149.90068 -36.83768)
10662,62985766,2024-08-14 13:27:14.580760960,-36.837437,-149.901382,1.185042e-15,36103,-9223372036854775808,7,NaN,NaN,01206H,"{'xmin': -149.90138244628906, 'ymin': -36.8374...",POINT (-149.90138 -36.83744)
10663,62985767,2024-08-14 13:27:14.596782720,-36.817005,-149.896744,3.884718e-15,36103,-9223372036854775808,7,NaN,NaN,01206H,"{'xmin': -149.89674377441406, 'ymin': -36.8170...",POINT (-149.89674 -36.81701)
